#### This notebook is used to compare and validate the data in experanto format with the data in the minimodel format

In [1]:
import os
import numpy as np
import torch


from minimodel import data
from minimodel import model_trainer_exp

import matplotlib.pyplot as plt
from omegaconf import OmegaConf, open_dict
from experanto.datasets import ChunkDataset
from experanto.dataloaders import get_multisession_dataloader

In [2]:
# minimodel

# setup
device = torch.device('cuda')
mouse_id = 0
data_path = '../data'
np.random.seed(1)

# load images
img = data.load_images(data_path, mouse_id, file=data.img_file_name[mouse_id])

# load neurons
fname = '%s_nat60k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id)
n_stim, n_neurons = spks.shape
print("spks_rep_all: ", spks_rep_all.shape)
print("spks: ", spks.shape)
# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)

# normalize data
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)


ineur = np.arange(0, n_neurons) #np.arange(0, n_neurons, 5)
spks_train = torch.from_numpy(spks[itrain][:,ineur]).to(device)
spks_val = torch.from_numpy(spks[ival][:,ineur]).to(device)

print('spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print('spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())
print('spks_test: ', spks_rep_all.shape, " with shape of a sample: ", spks_rep_all[0].shape)

img_train = torch.from_numpy(img[istim_train][itrain]).to(device).unsqueeze(1) # change :130 to 25:100 
img_val = torch.from_numpy(img[istim_train][ival]).to(device).unsqueeze(1)
img_test = img[istim_test]

print('img_train: ', img_train.shape, img_train.min(), img_train.max())
print('img_val: ', img_val.shape, img_val.min(), img_val.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())

raw image shape:  (68000, 66, 264)
cropped image shape:  (68000, 66, 130)
img:  (68000, 66, 130) -2.062947 2.088608 float32

loading activities from ../data/L1_A5_nat60k_2023_02_27.npz
spks_rep_all:  (500,)
spks:  (27533, 6636)

splitting training and validation set...
itrain:  (24779,)
ival:  (2754,)

normalizing neural data...
finished
spks_train:  torch.Size([24779, 6636]) tensor(0., device='cuda:0', dtype=torch.float64) tensor(58.7231, device='cuda:0', dtype=torch.float64)
spks_val:  torch.Size([2754, 6636]) tensor(0., device='cuda:0', dtype=torch.float64) tensor(49.2309, device='cuda:0', dtype=torch.float64)
spks_test:  (500,)  with shape of a sample:  (11, 6636)
img_train:  torch.Size([24779, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_val:  torch.Size([2754, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_test:  (500, 66, 130) -2.062947 2.088608


In [10]:
# Initialize lists to store reshaped spks_test and img_test
spks_test_list = []
img_test_list= []
# Iterate over each stimulus in the test set
print(img_test.dtype)
print(spks.dtype)
print(spks_rep_all[0].dtype)
for i, spks in enumerate(spks_rep_all):
    nrep, nneurons = spks.shape
    spks_test_list.append(spks)
    # Reshape the corresponding img_test
    img_rep = img_test[i][None, :, :].repeat(nrep, axis=0)
    img_test_list.append(img_rep)

spks_test_new = torch.from_numpy(np.concatenate(spks_test_list, axis=0))
img_test_new = torch.from_numpy(np.concatenate(img_test_list, axis=0))

print(img_test.dtype)
print(spks_test_new.dtype)

print("new:", spks_test_new.shape)
print("old:", spks_rep_all.shape)

print("new:", spks_test_new[0,0] )
print("old:", spks_rep_all[0][0,0])


float32
float32
float32
float32
torch.float32
new: torch.Size([4907, 6636])
old: (500,)
new: tensor(0.5784)
old: 0.5783657


In [11]:
print("new:", spks_test_new.shape)
print("old:", spks_rep_all.shape)

print("new:", spks_test_new[0,0] )
print("old:", spks_rep_all[0][0,0])

orig = spks_rep_all[0][0,0]
newv = spks_test_new[0,0].cpu().numpy()
print(orig, newv, float(orig) - float(newv))

new: torch.Size([4907, 6636])
old: (500,)
new: tensor(0.5784)
old: 0.5783657
0.5783657 0.5783657 0.0


In [4]:
# experanto

# --- setup ---
path_to_data = '/mnt/vast-nhr/projects/bthesis_cidas_richter/benjamin/minimodel/internship/data_experanto_testing'
data_folder = f'nat30k_{data.mouse_names[mouse_id]}_{data.exp_date[mouse_id]}_experanto'
data_path = os.path.join(path_to_data, data_folder)

# load configs for dataloaders
cfg_train = OmegaConf.load("./cfg_experanto/do_nothing_config.yaml")
cfg_val = OmegaConf.load("./cfg_experanto/do_nothing_config.yaml")
cfg_test = OmegaConf.load("./cfg_experanto/do_nothing_config.yaml")

cfg_train.dataset.modality_config.screen.valid_condition = {"tier": "train"}
cfg_train.dataset.out_keys.append("image_id")             # using this for debugging purposes
#cfg_train.dataloader.shuffle = False                     # using this for debugging purposes

cfg_train.dataloader.drop_last = False

cfg_val.dataset.modality_config.screen.valid_condition = {"tier": "validation"}
cfg_val.dataset.out_keys.append("image_id")             # using this for debugging purposes
#cfg_val.dataloader.shuffle = False                      # using this for debugging purposes
cfg_val.dataloader.drop_last = False

cfg_test.dataset.modality_config.screen.valid_condition = {"tier": "test"}
cfg_test.dataset.out_keys.append("image_id")        # I sadly need this to combine all samples with same image_id
cfg_test.dataloader.drop_last = False               # Here I dont need the batches to be the same size
#cfg_test.dataloader.shuffle = False

# build dataloaders
paths = [data_path]
train_dl = get_multisession_dataloader(paths, cfg_train)
val_dl = get_multisession_dataloader(paths, cfg_val)
test_dl = get_multisession_dataloader(paths, cfg_test)
print("Loaded experanto data from: ", data_path)

if cfg_train.dataloader.drop_last:  train_dl_length = len(train_dl) * cfg_train.dataloader.batch_size
else:                               train_dl_length = model_trainer_exp.count_samples(train_dl)
if cfg_val.dataloader.drop_last:    val_dl_length = len(val_dl) * cfg_val.dataloader.batch_size
else:                               val_dl_length = model_trainer_exp.count_samples(val_dl)
if cfg_test.dataloader.drop_last:   test_dl_length = len(test_dl) * cfg_test.dataloader.batch_size
else:                               test_dl_length =model_trainer_exp.count_samples(test_dl)

print("length of train_dl: ", train_dl_length)
print("length of val_dl: ", val_dl_length)
print("length of test_dl: ", test_dl_length)


/user/benjamin.richter02/u23846/.conda/envs/mini-exp-venv2/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


Loaded experanto data from:  /mnt/vast-nhr/projects/bthesis_cidas_richter/benjamin/minimodel/internship/data_experanto_testing/nat30k_L1_A5_022723_experanto
length of train_dl:  24779
length of val_dl:  2754
length of test_dl:  4906


In [ ]:
print("dataset len:", len(test_dl.loaders["session_0"].dataset))

dataset len: 4906


: 

In [ ]:
def compare_formats(data_exp, data_mini, modality="", tier="", only_diff=False):
    """ modality is either spks or img
    """
    data_exp = data_exp.to("cpu")
    data_mini = data_mini.to("cpu")


    print("=========== Now comparing data from modality:", modality, ", tier:", tier, " ===========")
    if not only_diff:
        # exp info
        print(modality+"_"+tier+"_exp: ", data_exp.shape, " min: ", data_exp.min().item(), " max: ", data_exp.max().item())
        data_exp_mean = torch.mean(data_exp)
        data_exp_std = torch.std(data_exp)
        print(modality+"_"+tier+" mean exp: ", data_exp_mean.cpu().numpy())
        print(modality+"_"+tier+" std exp: ", data_exp_std.cpu().numpy())
        print()
        # mini info:
        print(modality+"_"+tier+"_mini: ", data_mini.shape, " min: ", data_mini.min().item(), " max: ", data_mini.max().item())
        mini_spks_mean = torch.mean(data_mini)
        mini_spks_std = torch.std(data_mini)
        print(modality+"_"+tier+" mean mini: ", mini_spks_mean.cpu().numpy())
        print(modality+"_"+tier+" std mini: ", mini_spks_std.cpu().numpy())
        print()
        if modality=="spks":
            print("EXP samples: ", data_exp[:10,1])
            print("MINI samples: ", data_mini[:10,1])
        elif modality=="img":
            print("EXP samples: ", data_exp[1,0])
            print("MINI samples: ", data_mini[1,0])

    if data_exp.dtype != data_mini.dtype:
        data_exp = data_exp.to(dtype=data_mini.dtype) # dtype anpassen falls nötig


    numerical_same = torch.allclose(data_exp[:1000], data_mini[:1000], rtol=1e-5,atol=1e-8)  # atol: absolute tolerance, rtol relative tolerance: |a-b| <= atol + rtol*|b| for every element
    print("Tensors numericaly the same: ", numerical_same)
    diff = (data_exp - data_mini).abs()
    print(diff.shape)
    print("Total difference between " +modality+"_"+tier+" exp and " +modality+"_"+tier+" mini is: ", diff.sum().item())
    print("Mean difference per sample: ", diff.mean().item())
    print("Max difference for a sample: ", diff.max().item())

    rel_diff = diff / (data_mini.abs() + 1e-20)

    print("Max rel diff:", rel_diff.max().item())
    print("Mean rel diff:", rel_diff.mean().item())
    print()

In [5]:
batch_size = cfg_train.dataloader.batch_size

spks_train_exp = torch.zeros((train_dl_length, n_neurons), device=device)
img_train_exp = torch.zeros((train_dl_length, 1, 66, 130), device=device)
index_array = np.arange(0, train_dl_length, batch_size)
for k , (_, batch) in zip(index_array, train_dl):
    spks_batch = batch["responses"]
    img_batch = batch["screen"]
    kend = min(k+batch_size, train_dl_length)
    spks_train_exp[k:kend] = spks_batch.squeeze()
    img_batch = img_batch.squeeze().unsqueeze(1)        # shape: (batch_size, 1, 66,130)
    img_train_exp[k:kend] = img_batch



In [6]:

compare_formats(spks_train_exp, spks_train, modality="spks", tier="train", only_diff=True)
compare_formats(img_train_exp, img_train, modality="img", tier="train", only_diff=True)

=========== Now comparing data from modality: spks , tier: train  ===========
Tensors numericaly the same:  True
torch.Size([24779, 6636])
Total difference between spks_train exp and spks_train mini is:  1.9972630801817215
Mean difference per sample:  1.2146331254739891e-08
Max difference for a sample:  1.9031372175959405e-06
Max rel diff: 5.9597548594554816e-08
Mean rel diff: 1.3219101790750419e-08

=========== Now comparing data from modality: img , tier: train  ===========
Tensors numericaly the same:  True
torch.Size([24779, 1, 66, 130])
Total difference between img_train exp and img_train mini is:  0.0
Mean difference per sample:  0.0
Max difference for a sample:  0.0
Max rel diff: 0.0
Mean rel diff: 0.0



In [7]:
batch_size = cfg_val.dataloader.batch_size

spks_val_exp = torch.zeros((val_dl_length, n_neurons), device=device)
img_val_exp = torch.zeros((val_dl_length, 1, 66, 130), device=device)
index_array = np.arange(0, val_dl_length, batch_size)
for k , (_, batch) in zip(index_array, val_dl):
    spks_batch = batch["responses"]
    img_batch = batch["screen"]
    kend = min(k+batch_size, val_dl_length)
    spks_val_exp[k:kend] = spks_batch.squeeze()
    img_batch = img_batch.squeeze().unsqueeze(1)        # shape: (batch_size, 1, 66,130)
    img_val_exp[k:kend] = img_batch

In [8]:
compare_formats(spks_val_exp, spks_val, modality="spks", tier="val", only_diff=True)
compare_formats(img_val_exp, img_val, modality="img", tier="val", only_diff=True)

=========== Now comparing data from modality: spks , tier: val  ===========
Tensors numericaly the same:  True
torch.Size([2754, 6636])
Total difference between spks_val exp and spks_val mini is:  0.22075930575094713
Mean difference per sample:  1.2079492996265781e-08
Max difference for a sample:  1.7616739071968368e-06
Max rel diff: 5.95957931048501e-08
Mean rel diff: 1.3207067243229008e-08

=========== Now comparing data from modality: img , tier: val  ===========
Tensors numericaly the same:  True
torch.Size([2754, 1, 66, 130])
Total difference between img_val exp and img_val mini is:  0.0
Mean difference per sample:  0.0
Max difference for a sample:  0.0
Max rel diff: 0.0
Mean rel diff: 0.0



In [ ]:
batch_size = cfg_test.dataloader.batch_size

spks_test_exp = torch.zeros((test_dl_length, n_neurons), device=device)
img_test_exp = torch.zeros((test_dl_length, 1, 66, 130), device=device)
index_array = np.arange(0, test_dl_length, batch_size)
for k , (_, batch) in zip(index_array, test_dl):
    spks_batch = batch["responses"]
    img_batch = batch["screen"]
    kend = min(k+batch_size, test_dl_length)
    spks_test_exp[k:kend] = spks_batch.squeeze()
    img_batch = img_batch.squeeze().unsqueeze(1)        # shape: (batch_size, 1, 66,130)
    img_test_exp[k:kend] = img_batch


# Initialize lists to store reshaped spks_test and img_test
spks_test_list = []
img_test_list= []
# Iterate over each stimulus in the test set
for i, spks in enumerate(spks_rep_all):
    nrep, nneurons = spks.shape
    spks_test_list.append(spks)
    # Reshape the corresponding img_test
    img_rep = img_test[i][None, :, :].repeat(nrep, axis=0)
    img_test_list.append(img_rep)

spks_test = torch.from_numpy(np.concatenate(spks_test_list, axis=0))
img_test = torch.from_numpy(np.concatenate(img_test_list, axis=0))

    

: 

In [ ]:
#compare_formats(spks_test_exp, spks_test, modality="spks", tier="test", only_diff=True)
compare_formats(img_test_exp, img_test, modality="img", tier="test", only_diff=True)

=========== Now comparing data from modality: img , tier: test  ===========
